# Spider: дополнительные замеры участника 2

На тех же 1034 примерах оцениваем расширенный SFT, base с четырьмя примерами и Qwen2.5-0.5B-Instruct. Опубликованные base zero-shot и исходный SFT используются для сравнения. Включите GPU в Colab и загрузите подготовленный **spider_followup_inputs.zip** через панель Files в /content. Результаты сохраняются после каждого прогона, в конце скачивается ZIP.

Если загрузка пакета задерживается, ноутбук воссоздаёт точный тест и few-shot из Spider и запускает base few-shot и Instruct. Для расширенного SFT загрузите пакет и повторите ячейки подготовки и оценки. Готовые замеры переиспользуются в этой же сессии.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import torch

assert torch.cuda.is_available(), 'Select a GPU runtime'
print(torch.cuda.get_device_name(0), torch.__version__)
ROOT = Path('/content/spider-followup')
INPUT = Path('/content/spider_followup_inputs.zip')
SESSION = Path('/content/spider-followup-results')
REPO = 'https://github.com/Pavel-V418/BaseModel-vs-SFT.git'
EVALUATOR_COMMIT = '4ee0f7b5cdeea96c116ba610f347ea563d1507fd'
BASE = 'Qwen/Qwen2.5-0.5B'
INSTRUCT = 'Qwen/Qwen2.5-0.5B-Instruct'
REVISIONS = {
    BASE: '060db6499f32faf8b98477b0a26969ef7d8b9987',
    INSTRUCT: '7ae557604adf67be50417f59c2c2f167def9a775',
}
INPUT_SHA = 'f3c19975aa482b78902f1380f2e31daa79b7b61ed0f323abe9293eff8df1af16'
TEST_SHA = '07def41c02ff1bef775a2dd7ca0fc676b405d38b8e762d9bb32624e9aa328b03'
FEWSHOT_SHA = '25114622a92023c89514da93aa362db7a2372f16cf8ca61ee096792a283d321e'
ADAPTER = ROOT / 'outputs/sft_0.5b_nfull_others_r16_lr1e-4_s42_2ep/final'
ADAPTER_SHA = 'bd31c75f5d4e9adb470b9d3eed60fc935269b98982264a4363934c9976a7b9cd'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def sha256(path):
    with path.open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()

if not ROOT.exists():
    subprocess.run(['git', 'clone', REPO, str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', EVALUATOR_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-colab.txt')], check=True)

In [ ]:
def restore_eval_inputs():
    from src.data import make_records

    spider = ROOT / 'data/spider'
    archive = Path('/content/spider_data.zip')
    expected = '00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b'
    if not archive.is_file() or sha256(archive) != expected:
        url = 'https://huggingface.co/datasets/HAL-9001/spider-databases/resolve/main/spider_data.zip'
        download = archive.with_suffix('.download')
        subprocess.run(['curl', '-fL', '--retry', '2', '--max-time', '480', '-o', str(download), url], check=True)
        assert sha256(download) == expected, 'Spider archive differs from the original'
        download.replace(archive)

    spider.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for name in ['dev.json', 'tables.json', 'train_spider.json']:
            (spider / name).write_bytes(z.read('spider_data/' + name))
        raw = json.loads((spider / 'dev.json').read_text(encoding='utf-8'))
        for db in sorted({r['db_id'] for r in raw}):
            path = spider / 'database' / db / (db + '.sqlite')
            assert path.resolve().is_relative_to(spider)
            path.parent.mkdir(parents=True, exist_ok=True)
            member = 'spider_data/' + path.relative_to(spider).as_posix()
            with z.open(member) as source, path.open('wb') as target:
                shutil.copyfileobj(source, target)

    train = json.loads((spider / 'train_spider.json').read_text(encoding='utf-8'))
    examples = [train[i] for i in (18, 2749, 1835, 2897)]
    for name, examples_raw, output in [('test', raw, TEST), ('fewshot', examples, FEWSHOT)]:
        prepared, _ = make_records(examples_raw, name, spider, check_exec=False)
        with output.open('w', encoding='utf-8', newline='') as stream:
            for row in prepared:
                stream.write(json.dumps(row, ensure_ascii=False) + chr(13) + chr(10))

TEST = ROOT / 'data/test.jsonl'
FEWSHOT = ROOT / 'data/fewshot.jsonl'
if INPUT.is_file() and INPUT.stat().st_size == 64714640:
    assert sha256(INPUT) == INPUT_SHA
    with zipfile.ZipFile(INPUT) as archive:
        for item in archive.infolist():
            target = (ROOT / item.filename).resolve()
            assert target.is_relative_to(ROOT / 'data') or target.is_relative_to(ADAPTER)
        archive.extractall(ROOT)
elif not TEST.is_file() or not FEWSHOT.is_file():
    sys.path.insert(0, str(ROOT))
    restore_eval_inputs()
assert sha256(TEST) == TEST_SHA
assert sha256(FEWSHOT) == FEWSHOT_SHA
adapter_ready = (ADAPTER / 'adapter_model.safetensors').is_file()
if adapter_ready:
    assert sha256(ADAPTER / 'adapter_model.safetensors') == ADAPTER_SHA
rows = [json.loads(line) for line in TEST.read_text(encoding='utf-8').splitlines()]
assert len(rows) == 1034
SESSION.mkdir(exist_ok=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_parse.py', 'tests/test_evaluate.py'], cwd=ROOT, check=True)

from huggingface_hub import snapshot_download
from importlib.metadata import version
from transformers import AutoTokenizer

def checkpoint(model):
    return snapshot_download(
        repo_id=model, revision=REVISIONS[model],
        allow_patterns=['*.json', '*.safetensors', 'vocab.*', 'merges.*', '*.jinja'],
    )

def run_eval(name, model, mode, *, adapter=None, data=TEST, chat_template=False):
    metrics_path = SESSION / f'{name}_eval_metrics.json'
    predictions_path = SESSION / f'{name}_eval.jsonl'
    if metrics_path.exists() and predictions_path.exists():
        saved = json.loads(metrics_path.read_text())
        if saved.get('dataset_sha256') == TEST_SHA and saved.get('model_revision') == REVISIONS[model] and saved.get('adapter_sha256') == (ADAPTER_SHA if adapter else None) and saved.get('predictions_sha256') == sha256(predictions_path) and saved.get('evaluator_commit') == EVALUATOR_COMMIT and sum(1 for _ in predictions_path.open(encoding='utf-8')) == len(rows):
            print('Reusing completed run:', name)
            return
    local = checkpoint(model)
    if chat_template:
        tokenizer = AutoTokenizer.from_pretrained(local)
        data = Path('/content/test_instruct.jsonl')
        with data.open('w', encoding='utf-8') as target:
            for row in rows:
                prompt = tokenizer.apply_chat_template(
                    [{'role': 'user', 'content': row['prompt']}],
                    tokenize=False, add_generation_prompt=True,
                )
                target.write(json.dumps({**row, 'prompt': prompt}, ensure_ascii=False) + chr(10))
    argv = [
        sys.executable, '-u', '-m', 'src.evaluate', '--model', local,
        '--data', str(data), '--fewshot', str(FEWSHOT), '--mode', mode,
        '--run_name', name, '--results_dir', str(SESSION),
    ]
    if adapter is not None:
        argv += ['--adapter', str(adapter)]
    print('Starting:', name, flush=True)
    with (SESSION / f'{name}.log').open('w', encoding='utf-8') as log:
        with subprocess.Popen(argv, cwd=ROOT, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8') as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
        if process.returncode:
            raise subprocess.CalledProcessError(process.returncode, argv)
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    assert metrics['num_examples'] == len(rows)
    metrics.update(
        model=model, model_revision=REVISIONS[model],
        adapter=adapter.relative_to(ROOT).as_posix() if adapter else None,
        adapter_sha256=ADAPTER_SHA if adapter else None,
        chat_template=chat_template, dataset_sha256=TEST_SHA, input_sha256=INPUT_SHA if adapter else None,
        rendered_dataset_sha256=sha256(data),
        fewshot_sha256=FEWSHOT_SHA if mode == 'few' else None,
        evaluator_commit=EVALUATOR_COMMIT, gpu=torch.cuda.get_device_name(0),
        versions={p: version(p) for p in ['torch', 'transformers', 'peft', 'accelerate']},
        predictions_sha256=sha256(predictions_path),
    )
    metrics_path.write_text(json.dumps(metrics, indent=2) + chr(10), encoding='utf-8')
    print('Complete:', name, metrics['overall']['execution_accuracy'], flush=True)

In [ ]:
if adapter_ready:
    run_eval('sft_0.5b_expanded_zero', BASE, 'zero', adapter=ADAPTER)
else:
    print('Expanded SFT is waiting for the adapter package')

In [ ]:
run_eval('base_0.5b_few', BASE, 'few')

In [ ]:
run_eval('instruct_0.5b_zero', INSTRUCT, 'zero', chat_template=True)

In [ ]:
names = [
    ('base_0.5b_zero', ROOT / 'results'),
    ('sft_0.5b_zero', ROOT / 'results'),
    ('sft_0.5b_expanded_zero', SESSION),
    ('base_0.5b_few', SESSION),
    ('instruct_0.5b_zero', SESSION),
]
comparison = {}
for name, folder in names:
    path = folder / f'{name}_eval_metrics.json'
    if not path.is_file():
        continue
    result = json.loads(path.read_text(encoding='utf-8'))
    assert result['num_examples'] == len(rows)
    comparison[name] = result['overall']
    print(name, f"{result['overall']['execution_accuracy']:.2%}")
(SESSION / 'comparison.json').write_text(json.dumps(comparison, indent=2) + chr(10))
archive = shutil.make_archive('/content/spider_followup_results', 'zip', SESSION)
from google.colab import files
files.download(archive)